# 🏷️ Modul 04: Supervised Learning I - Klasifikasi (*Classification*)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/04_Classification_Algorithms.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Target Audiens**: Pemula tanpa latar belakang matematika/statistika tingkat lanjut.  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & analogi dunia nyata sebelum menyentuh baris kode algoritma.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── [5. Evaluation] ◄── 🎯 [4. MODELING: KLASIFIKASI]
```
Di Modul 01-03, Anda telah membersihkan dan mengeksplorasi data. Sekarang, di **Modul 04**, kita memasuki jantung keajaiban *Data Mining* & *Machine Learning*: **Membuat mesin belajar mengambil keputusan sendiri (*Supervised Classification*)**!


## 🧠 1. Filosofi Supervised Learning: Mengajari Mesin dari Masa Lalu

### Analogi Ramah Awam: Murid dan Buku Latihan Berisi Kunci Jawaban
Bayangkan seorang murid yang sedang belajar untuk ujian:
1. **Fitur ($X$)**: Soal-soal latihan (misal: ciri-ciri pasien seperti tekanan darah, umur, dan kadar kolesterol).
2. **Target Label ($y$)**: Kunci jawaban di halaman belakang buku (misal: "Kanker Ganas" atau "Tumor Jinak").

Dalam **Supervised Learning (Pembelajaran Terawasi)**, kita menyodorkan ribuan pasang $(X, y)$ ke algoritma komputer. Algoritma akan mencari pola tersembunyi yang menghubungkan $X$ ke $y$. Begitu pola tersebut dikuasai, kita memberikan pasien baru yang belum pernah ada kunci jawabannya ($X_{\text{baru}}$), dan mesin dapat memprediksi labelnya ($y_{\text{pred}}$) secara akurat!

### 🆚 Klasifikasi vs Regresi: Apa Bedanya?
* **Klasifikasi (*Classification*)**: Target $y$ berbentuk **Kategori / Label Diskrit**.  
  *Contoh*: Menentukan email Spam vs Bukan Spam, Prediksi Churn Pelanggan (Keluar / Bertahan), Diagnosis Medis (Positif / Negatif).
* **Regresi (*Regression*)**: Target $y$ berbentuk **Angka Kontinu**.  
  *Contoh*: Memprediksi harga rumah (Rp 850 juta), estimasi omzet penjualan bulanan, prediksi suhu cuaca besok.


## ✂️ 2. Partisi Data: Train vs Test, Stratifikasi, & Bahaya Overfitting

### 🎓 Analogi Ujian Akhir Semester
Jika seorang guru mengajar menggunakan 100 soal latihan, lalu saat Ujian Akhir Semester guru tersebut mengeluarkan 100 soal yang **persis sama**, apakah nilai 100 murid membuktikan bahwa murid tersebut pintar?  
**Tentu tidak!** Murid tersebut bisa saja hanya **menghafal kunci jawaban** tanpa memahami konsep sama sekali (*Overfitting*).

Oleh karena itu, dalam ilmu data, kita wajib membelah data menjadi dua bagian:
1. **Data Latih (*Training Set* ~ 70-80%)**: Diberikan kepada model untuk belajar pola.
2. **Data Uji (*Testing Set* ~ 20-30%)**: Disimpan rapat di brankas! Hanya dikeluarkan untuk menguji model di akhir, seperti soal ujian yang belum pernah dilihat sebelumnya.

### ⚖️ Parameter `stratify=y`: Menjaga Keseimbangan Sampel
Jika pada populasi asli terdapat 90% pasien sehat dan 10% pasien kanker, maka baik *Data Latih* maupun *Data Uji* wajib mempertahankan rasio 90:10 yang sama persis. Parameter `stratify=y` memastikan pembagian kelas berlangsung adil dan representatif.


In [ ]:
# Import pustaka fundamental
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Memuat dataset standar medis terverifikasi: Breast Cancer Wisconsin
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target') # 0 = Ganas (Malignant), 1 = Jinak (Benign)

print(f"Dimensi Data: {X.shape[0]} baris × {X.shape[1]} fitur klinis")
print("Distribusi Target Medis:")
print(f" - Kanker Ganas (0 - Malignant): {(y == 0).sum()} pasien ({(y == 0).mean()*100:.1f}%)")
print(f" - Tumor Jinak   (1 - Benign)   : {(y == 1).sum()} pasien ({(y == 1).mean()*100:.1f}%)")

# Partisi Data Latih & Uji (80% Train, 20% Test) dengan STRATIFIKASI
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling: WAJIB fit hanya pada X_train untuk mencegah DATA LEAKAGE!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nJumlah Data Latih (Train Set): {X_train.shape[0]} baris")
print(f"Jumlah Data Uji   (Test Set) : {X_test.shape[0]} baris")


## 🥊 3. Menjelajah 4 Algoritma Klasifikasi Populer

Mari kita bedah 4 algoritma klasifikasi paling terkenal dalam sejarah data mining, dari intuisi hingga kode implementasinya:

---

### A. K-Nearest Neighbors (k-NN)
* **Filosofi**: *"Katakan padaku siapa 5 teman terdekatmu, maka aku akan tahu siapa dirimu."*
* **Cara Kerja**: Menghitung jarak geometris (biasanya jarak **Euclidean** $\sqrt{\sum (x_i - y_i)^2}$) antara data baru dengan seluruh data di memori. Model memilih $K$ tetangga paling dekat, lalu mengambil suara terbanyak (*majority vote*).
* **Aturan Emas**:
  1. Nilai $K$ sebaiknya **ganjil** (misal: 3, 5, 7) agar tidak terjadi hasil imbang (seri) pada klasifikasi biner.
  2. k-NN **WAJIB menggunakan Feature Scaling**! Jika ada fitur bergaji Rp 10.000.000 dan usia 25 tahun, jarak gaji akan mendominasi dan membutakan usia.

---

### B. Naive Bayes (Gaussian Naive Bayes)
* **Filosofi**: Dokter yang mengombinasikan probabilitas gejala demam dan batuk menggunakan rumus Bayes:
  $$P(\text{Sakit} | \text{Gejala}) = \frac{P(\text{Gejala} | \text{Sakit}) \times P(\text{Sakit})}{P(\text{Gejala})}$$
* **Mengapa disebut "Naif"?**: Karena algoritma ini mengasumsikan bahwa seluruh fitur bersifat **saling independen** (tidak saling memengaruhi). Di dunia nyata batuk dan demam sering terjadi bersamaan, namun asumsi naif ini justru membuat Naive Bayes **sangat cepat, ringan memori, dan handal untuk baseline maupun klasifikasi teks/spam**.

---

### C. Decision Tree (Pohon Keputusan)
* **Filosofi**: Bagan alur flowchart panduan diagnosis SOP dokter: *"Apakah ukuran tumor > 15mm? Jika Ya ➔ Apakah tekstur > 20? ➔ Jika Ya: Kanker Ganas"*.
* **Cara Memilih Cabang**: Menggunakan ukuran kemurnian data: **Gini Impurity** (0 = murni sempurna satu kelas, 0.5 = seimbang campur aduk) atau **Entropy / Information Gain**.
* **Kelebihan**: Sangat mudah dijelaskan ke manajemen bisnis (*White Box Model*).
* **Kelemahan**: Jika dibiarkan tumbuh bebas tanpa batas (`max_depth`), pohon akan menghafal seluruh data hingga daunnya hanya berisi 1 sampel (*Severe Overfitting*).

---

### D. Random Forest (Ensemble Bagging)
* **Filosofi**: *"Kebijaksanaan Orang Banyak (Wisdom of the Crowds)"*. Jangan hanya meminta diagnosis 1 orang dokter. Kumpulkan 100 dokter spesialis independen, berikan setiap dokter sudut pandang berbeda, lalu ambil keputusan akhir lewat **Voting Terbanyak**.
* **Teknik Rahasia**:
  1. **Bootstrap Aggregating (Bagging)**: Setiap pohon dilatih menggunakan sampel acak dengan pengembalian.
  2. **Feature Randomness**: Setiap cabang pohon hanya diizinkan memilih sebagian kecil fitur secara acak, sehingga pohon-pohon yang dihasilkan saling berbeda dan tidak seragam.
* **Hasil**: Salah satu algoritma klasifikasi paling akurat, stabil, dan tangguh di industri.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Inisialisasi ke-4 model
models = {
    "K-Nearest Neighbors (K=5)": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes (GaussianNB)": GaussianNB(),
    "Decision Tree (Max Depth=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest (100 Trees)": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

hasil_akurasi = {}

print("=== PELATIHAN DAN AKURASI AWAL ===")
for nama_model, model in models.items():
    # Catatan: k-NN wajib data berskala (scaled), tree-based aman dengan atau tanpa scale
    if "Neighbors" in nama_model:
        model.fit(X_train_scaled, y_train)
        pred_test = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        pred_test = model.predict(X_test)
        
    acc = accuracy_score(y_test, pred_test)
    hasil_akurasi[nama_model] = acc
    print(f"✅ {nama_model:<30} : {acc*100:.2f}%")


In [ ]:
# Visualisasi Struktur Pohon Keputusan (Decision Tree)
dt_model = models["Decision Tree (Max Depth=3)"]

plt.figure(figsize=(16, 7), dpi=150)
plot_tree(
    dt_model,
    feature_names=data.feature_names,
    class_names=['Ganas (0)', 'Jinak (1)'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Visualisasi Struktur Pohon Keputusan (Decision Tree)", fontsize=14, fontweight='bold', pad=15)
plt.show()

print("Interpretasi Diagram Pohon:")
print("1. Kotak paling atas (Root Node) adalah fitur pemilah paling krusial yang ditemukan algoritma.")
print("2. Warna jingga = Dominan Kanker Ganas (0), Warna biru = Dominan Tumor Jinak (1).")
print("3. Semakin pekat warnanya, semakin murni (rendah Gini Impurity) sampel di node tersebut!")


## 📊 4. Evaluasi Mendalam: Mengapa Akurasi Saja Berbahaya? (*The Accuracy Paradox*)

### ⚠️ Bahaya Jebakan "Akurasi 99%"
Bayangkan kasus deteksi transaksi perbankan mencurigakan (*Fraud Detection*):
* Dari 10.000 transaksi harian, hanya ada 10 transaksi penipuan (0.1%).
* Jika seorang programmer malas membuat model bodoh yang selalu memprediksi **"SEMUA TRANSAKSI AMAN"**:
  $$\text{Akurasi} = \frac{9.990}{10.000} = 99.9\%$$
* Akurasinya luar biasa tinggi (99.9%)! Namun sistem tersebut **gagal menangkap 10 penipuan**, dan bank merugi miliaran rupiah!

Inilah alasan mengapa praktisi data senior **tidak pernah hanya mengandalkan Akurasi**. Kita wajib membedah **Confusion Matrix**, **Precision**, dan **Recall**.

---

### 🗂️ Membedah Anatomi Confusion Matrix
| Istilah Medis | Makna Nyata | Analogi Alarm Kebakaran |
| :--- | :--- | :--- |
| **True Positive (TP)** | Pasien Kanker diprediksi Kanker | Ada kebakaran, alarm berbunyi ✅ |
| **True Negative (TN)** | Pasien Sehat diprediksi Sehat | Aman damai, alarm hening ✅ |
| **False Positive (FP)** | Pasien Sehat diprediksi Kanker (*Type I Error*) | Goreng ikan bakar, alarm bunyi ⚠️ |
| **False Negative (FN)** | Pasien Kanker diprediksi Sehat (*Type II Error*) | **Gedung terbakar, alarm mati! (FATAL)** 🚨 |

---

### ⚖️ Kapan Harus Precision vs Kapan Recall?
1. **Prioritaskan RECALL**: $\frac{TP}{TP + FN}$  
   * **Tujuan**: Meminimalkan **False Negative (FN)** sampai nol. Kita tidak boleh membiarkan satu pun orang sakit lolos dengan label sehat.  
   * *Contoh*: Deteksi Kanker, Cacat Komponen Mesin Pesawat Terbang, Gempa Bumi.
2. **Prioritaskan PRECISION**: $\frac{TP}{TP + FP}$  
   * **Tujuan**: Meminimalkan **False Positive (FP)**. Kita tidak ingin alarm palsu merugikan pengguna atau bisnis.  
   * *Contoh*: Filter Spam Email (email penting kantor jangan sampai masuk folder spam), Keputusan Menahan Dana Nasabah.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score

# Kita ambil model Random Forest sebagai representasi evaluasi
rf_model = models["Random Forest (100 Trees)"]
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1] # Probabilitas kelas 1 (Jinak)

# 1. Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ganas (0)', 'Jinak (1)'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title("Confusion Matrix - Random Forest", fontweight='bold', pad=12)
plt.grid(False)
plt.show()

# 2. Cetak Classification Report Lengkap
print("=== LAPORAN EVALUASI MODEL (CLASSIFICATION REPORT) ===")
print(classification_report(y_test, y_pred_rf, target_names=['Ganas (0)', 'Jinak (1)']))


In [ ]:
# Membandingkan Kurva ROC (Receiver Operating Characteristic) & Skor AUC ke-4 Model
plt.figure(figsize=(8, 6))

for nama_model, model in models.items():
    if "Neighbors" in nama_model:
        y_probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_probs = model.predict_proba(X_test)[:, 1]
        
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    score_auc = roc_auc_score(y_test, y_probs)
    plt.plot(fpr, tpr, lw=2, label=f"{nama_model} (AUC = {score_auc:.3f})")

# Garis tebakan acak (Random Guess Baseline = 0.5)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label="Tebakan Acak (AUC = 0.500)")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Recall / Sensitivity)", fontsize=11)
plt.title("Perbandingan Kurva ROC (Receiver Operating Characteristic)", fontsize=13, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print("💡 Cara Membaca Kurva ROC & Skor AUC:")
print("1. Sumbu X: Kesalahan mendiagnosis orang sehat sebagai kanker (False Positive Rate).")
print("2. Sumbu Y: Keberhasilan mendeteksi orang yang benar-benar sakit (True Positive Rate).")
print("3. Semakin kurva melengkung mendekati pojok kiri atas (AUC mendekati 1.0), semakin hebat performa model memisahkan kedua kelas!")


## ⚖️ 5. Menangani Kasus Data Tidak Seimbang (*Imbalanced Class*) dengan SMOTE

### Masalah Nyata di Industri
Seringkali di dunia nyata, kasus yang kita cari adalah kejadian langka (*Minority Class*):
* Kasus penipuan kartu kredit (0.2%).
* Pasien penderita penyakit langka (2%).
* Pelanggan yang membatalkan langganan / churn (5%).

Jika model dilatih pada rasio ekstrem ini, model akan malas dan cenderung mengabaikan kelas minoritas.

### Solusi Modern: SMOTE (*Synthetic Minority Over-sampling Technique*)
* **Bukan Sekadar Duplikasi**: Teknik lama (*Random Over-sampling*) hanya menggandakan baris data minoritas yang sama berulang-ulang, yang memicu *Overfitting*.
* **Sintesis Cerdas**: SMOTE memilih dua titik data minoritas terdekat, menarik garis maya di antara keduanya, lalu **menciptakan titik sintetis baru** di sepanjang garis tersebut!


In [ ]:
# Otomatisasi instalasi dan impor imblearn
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn", "-q"])
    from imblearn.over_sampling import SMOTE

# Simulasi Kasus Ekstrem: Mari kita buang 90% data Kanker Ganas agar data menjadi sangat tidak seimbang
mask_pertahankan = (y_train == 1) | ((y_train == 0) & (np.random.RandomState(42).rand(len(y_train)) < 0.15))
X_imb = X_train[mask_pertahankan]
y_imb = y_train[mask_pertahankan]

print("=== DISTRIBUSI KELAS TIDAK SEIMBANG (SIMULASI IMBALANCED) ===")
print(f"Jumlah Kanker Ganas (Minoritas): {(y_imb == 0).sum()} sampel")
print(f"Jumlah Tumor Jinak  (Mayoritas): {(y_imb == 1).sum()} sampel")

# Menerapkan SMOTE untuk menyeimbangkan kelas minoritas
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_imb, y_imb)

print("\n=== DISTRIBUSI SETELAH DITERAPKAN SMOTE ===")
print(f"Jumlah Kanker Ganas (Minoritas): {(y_resampled == 0).sum()} sampel (Sintetis diciptakan)")
print(f"Jumlah Tumor Jinak  (Mayoritas): {(y_resampled == 1).sum()} sampel")

# Visualisasi Perbandingan Sebelum vs Sesudah SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Ganas (0)', 'Jinak (1)'], [(y_imb == 0).sum(), (y_imb == 1).sum()], color=['salmon', 'skyblue'])
axes[0].set_title(f"Sebelum SMOTE (Total: {len(y_imb)})", fontweight='bold')
axes[0].set_ylabel("Jumlah Pasien")

axes[1].bar(['Ganas (0)', 'Jinak (1)'], [(y_resampled == 0).sum(), (y_resampled == 1).sum()], color=['salmon', 'skyblue'])
axes[1].set_title(f"Sesudah SMOTE (Total: {len(y_resampled)})", fontweight='bold')

plt.tight_layout()
plt.show()


## 🧠 6. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep inti klasifikasi. Jalankan sel kode di bawah untuk mencocokkan jawaban Anda dengan kunci jawaban logis!

---
* **Soal 1**: Mengapa pada klasifikasi biner menggunakan k-NN, kita sangat dianjurkan memilih nilai $K$ yang ganjil (misal $K=3$ atau $5$)?
* **Soal 2**: Dalam sistem deteksi penyakit kritis mematikan (seperti kanker), metrik mana yang paling wajib dimaksimalkan: **Precision** atau **Recall**? Mengapa?
* **Soal 3**: Mengapa kita wajib melakukan `scaler.fit()` HANYA pada data latih (`X_train`), dan tidak boleh pada seluruh dataset sekaligus sebelum di-split?
* **Soal 4**: Apa perbedaan mendasar antara model tunggal *Decision Tree* dan ensemble *Random Forest*?


In [ ]:
# Jalankan sel ini untuk meninjau kunci jawaban evaluasi mandiri
kunci_jawaban_modul_4 = {
    "Soal 1": "Memilih K ganjil mencegah terjadinya voting imbang (seri/tie-break) antara 2 kelas.",
    "Soal 2": "RECALL wajib dimaksimalkan. Sebab False Negative (orang sakit diprediksi sehat) berakibat fatal bagi nyawa pasien tanpa penanganan.",
    "Soal 3": "Mencegah Data Leakage (kebocoran informasi masa depan dari data uji ke data latih yang membuat performa evaluasi bias/menipu).",
    "Soal 4": "Decision Tree adalah 1 pohon tunggal yang rentan overfit, sedangkan Random Forest menggabungkan puluhan/ratusan pohon independen lewat voting terbanyak (Bagging) sehingga jauh lebih stabil."
}

print("=== KUNCI JAWABAN & PEMBAHASAN EVALUASI MANDIRI ===")
for soal, pembahasan in kunci_jawaban_modul_4.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 7. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Mencari Nilai $K$ Terbaik pada k-NN
Pada algoritma k-NN:
* Jika $K$ terlalu kecil ($K=1$), model sangat sensitif terhadap noise (*Overfitting*).
* Jika $K$ terlalu besar ($K=50$), model menggeneralisir terlalu kasar (*Underfitting*).

Tugas Anda:
1. Ujilah variasi nilai $K = [1, 3, 5, 7, 9, 11, 15, 21, 31]$.
2. Catat akurasi data uji (*Test Accuracy*) untuk setiap nilai $K$.
3. Gambarkan grafik garis hubungan antara nilai $K$ dan akurasi untuk menemukan **nilai $K$ optimal**!


In [ ]:
# Solusi Eksperimen Nilai K Optimal
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31]
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_scores.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

# Visualisasi Elbow / Optimal K Curve
plt.figure(figsize=(9, 4.5))
plt.plot(k_values, train_scores, marker='o', label='Akurasi Data Latih (Train Set)', color='royalblue')
plt.plot(k_values, test_scores, marker='s', label='Akurasi Data Uji (Test Set)', color='crimson', lw=2)

best_k = k_values[np.argmax(test_scores)]
best_score = max(test_scores)

plt.axvline(x=best_k, color='green', linestyle='--', label=f'K Optimal = {best_k} ({best_score*100:.1f}%)')
plt.title("Pencarian Nilai K Optimal pada k-NN (Bias-Variance Tradeoff)", fontsize=13, fontweight='bold')
plt.xlabel("Nilai K (Jumlah Tetangga Terdekat)")
plt.ylabel("Akurasi Model")
plt.xticks(k_values)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print(f"🏆 Kesimpulan Eksperimen: Nilai K terbaik adalah K = {best_k} dengan akurasi uji {best_score*100:.2f}%!")


## 🎯 8. Checklist Kelulusan Modul 04 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami perbedaan mendasar problem **Klasifikasi** (kategori) vs **Regresi** (angka kontinu).
- [ ] Saya paham mengapa pembagian data `train_test_split` dan parameter `stratify=y` sangat penting untuk mencegah bias.
- [ ] Saya memahami intuisi kerja 4 algoritma: **k-NN** (jarak tetangga), **Naive Bayes** (probabilitas independen), **Decision Tree** (bagan alur Gini), dan **Random Forest** (ensemble 100 dokter).
- [ ] Saya bisa menjelaskan bahaya *Accuracy Paradox* dan mengapa akurasi 99% bisa menipu pada kasus data timpang (*imbalanced*).
- [ ] Saya dapat membaca diagram **Confusion Matrix**, menghitung **Precision**, **Recall**, dan **F1-Score**.
- [ ] Saya mengerti cara kerja **SMOTE** dalam menciptakan sampel minoritas sintetis.

---
🚀 **Luar Biasa! Anda telah menaklukkan Modul 04!**  
Di modul berikutnya (**Modul 05**), kita akan menjelajahi **Supervised Learning II: Regresi**, di mana kita akan memprediksi angka kontinu di masa depan menggunakan regresi linear hingga regresi non-linear ensemble!
